In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth
import pyspark.sql.functions as F

# Initialize local Spark
spark = SparkSession.builder \
    .appName("Pediatric_ADR_Mining_Local") \
    .config("spark.driver.memory", "18g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/13 01:11:24 WARN Utils: Your hostname, localhost.localdomain, resolves to a loopback address: 127.0.0.1; using 192.168.1.17 instead (on interface wlp0s20f3)
26/01/13 01:11:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/13 01:11:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.parquet("./2025q3_cleansed.parquet")

In [3]:
# FP-Growth at the "Signal Detection" threshold
fpGrowth = FPGrowth(
    itemsCol="adr_case", 
    minSupport=0.0005,
    minConfidence=0.9    # Lower confidence to catch more associations
)

model = fpGrowth.fit(df)

In [4]:
# Get the rules
rules = model.associationRules

In [5]:
# IMPORTANT: Filter for actual insights (Consequents should be Reactions)
# This removes the "noise" of demographic-to-demographic rules
interesting_rules = rules.filter(
    F.array_join("consequent", ",").contains("reaction_") | 
    F.array_join("consequent", ",").contains("outc_")
).sort(F.desc("lift"))

In [6]:
interesting_rules.show(20, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------+----------+------------------+--------------------+
|antecedent                                                                                                                                                                                                                                                   |consequent                                  |confidence|lift              |support             |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------

In [15]:
interesting_rules.count()

1701186

In [7]:
interesting_rules.write.mode('overwrite').format('parquet').save('mined_rules.parquet')

In [9]:
import pandas as pd

df = pd.read_parquet('mined_rules/')
print(len(df))

1701186


In [10]:
df.head(5)

,antecedent,consequent,confidence,lift,support
0,"[reaction_chronic_gastrointestinal_bleeding, r...",[reaction_peptic_ulcer],1.0,1945.428571,0.000514
1,"[reaction_peptic_ulcer, reaction_medication_ov...",[reaction_chronic_gastrointestinal_bleeding],1.0,1945.428571,0.000514
2,"[reaction_chronic_gastrointestinal_bleeding, r...",[reaction_peptic_ulcer],1.0,1945.428571,0.000514
3,"[reaction_chronic_gastrointestinal_bleeding, r...",[reaction_peptic_ulcer],1.0,1945.428571,0.000514
4,"[reaction_peptic_ulcer, reaction_medication_ov...",[reaction_chronic_gastrointestinal_bleeding],1.0,1945.428571,0.000514


In [12]:
import duckdb

conn = duckdb.connect('mined_rules.duckdb')

query = """
create table mined_rules.main.rules as 
select * from df limit 0
"""

conn.execute(query)

In [13]:
query = """
insert into mined_rules.main.rules
select * from df
"""

conn.execute(query)

In [14]:
conn.execute('select count(*) from mined_rules.main.rules').fetchone()

(1701186,)